# ZemaVoice — Stage 2: Amharic TTS dataset preparation
**Free Colab notebook.** Downloads up to 120 FLEURS Amharic training examples, audits transcripts and recordings, previews audio, and exports a training-ready manifest. This is **not yet a model training run**.
FLEURS is CC BY 4.0; retain attribution and review the dataset card, audio quality, and voice rights before commercial synthesis. Do not clone recognizable speakers without permission.


In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
!pip -q install 'datasets[audio]>=3,<5' soundfile numpy


In [ ]:
from datasets import load_dataset
import soundfile as sf, numpy as np, json, os
from pathlib import Path
from IPython.display import Audio, display
ROOT=Path('/content/zemavoice_amharic')
(ROOT/'audio').mkdir(parents=True,exist_ok=True)
MAX_CLIPS=120
print('Streaming licensed Amharic FLEURS training split...')
ds=load_dataset('google/fleurs','am_et',split='train',streaming=True)
records=[]
for item in ds:
    text=(item.get('transcription') or '').strip()
    if not text or not any('\u1200'<=c<='\u137f' for c in text): continue
    audio=item['audio']; samples=np.asarray(audio['array'],dtype=np.float32)
    sr=int(audio['sampling_rate']); duration=len(samples)/sr
    if not (1.0 <= duration <= 25.0): continue
    if not np.isfinite(samples).all() or np.max(np.abs(samples)) < 0.001: continue
    path=ROOT/'audio'/f'am_{len(records):04d}.wav'
    sf.write(str(path),samples,sr)
    records.append({'audio_path':str(path),'transcript':text,'duration_seconds':round(duration,3),'sample_rate':sr,'speaker_id':'unknown_fleurs','source':'google/fleurs am_et','license':'CC-BY-4.0','split':'train','consent_record':'dataset-license-not-individual-cloning-consent'})
    if len(records)>=MAX_CLIPS: break
with (ROOT/'manifest.jsonl').open('w',encoding='utf-8') as f:
    for rec in records: f.write(json.dumps(rec,ensure_ascii=False)+'\n')
print('Prepared clips:',len(records))
print('Total minutes:',round(sum(x['duration_seconds'] for x in records)/60,2))
print('Manifest:',ROOT/'manifest.jsonl')


In [ ]:
assert records, 'No recordings were prepared; inspect dataset loading errors above.'
for i in range(min(3,len(records))):
    rec=records[i]
    print('Clip',i+1,'—',rec['transcript'])
    display(Audio(filename=rec['audio_path']))
print('Listen and compare with transcripts before considering training.')


In [ ]:
from google.colab import files
import zipfile
archive='/content/zemavoice_amharic_sample.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob('*'):
        if p.is_file(): z.write(p,arcname=str(p.relative_to(ROOT)))
print('Created',archive)
print('To download, uncomment the next line:')
# files.download(archive)


## After this notebook
The clips and manifest are saved in temporary Colab storage. Download the ZIP if you want to keep them. A model is **not** trained by this notebook. The next engineering task is selecting a commercially compatible Amharic TTS architecture and running a first GPU training experiment on consent-cleared, quality-checked speech. Colab free sessions can expire and do not guarantee training completion.
